In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat

In [2]:
recording_name = '20251212_128CHS_V1_1_26g_251212_182717'
channel_list = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [3]:
file_list = os.listdir(f"/home/ubuntu/Documents/jct/project/20251212-LN LYQ CA1 V1/{recording_name}")
file_list.remove("settings.xml")

file_list = sorted(file_list)
recording_raw_list = []
for file in file_list:
    recording_raw_list.append(se.read_intan(f"/home/ubuntu/Documents/jct/project/20251212-LN LYQ CA1 V1/{recording_name}/{file}", stream_id= '0'))
recording_raw = concatenate_recordings(recording_list=recording_raw_list)
recording_raw = recording_raw.select_channels(channel_list)

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [4]:
traces = recording_f.get_traces(start_frame = 1000, end_frame= 300 * 20000, channel_ids=[
       'B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019'])

In [ ]:
# ============================================================================
# 生成20s窗口的PDF（100秒数据，每个窗口20秒）
# ============================================================================

print("\n" + "="*60)
print("生成20s窗口PDF (100s数据, 每个窗口20s)")
print("="*60)

# 使用已读取的traces数据
assert 'traces' in locals() or 'traces' in globals(), "请先读取traces数据"
assert traces.ndim == 2, f"traces应该是2维数组，当前形状: {traces.shape}"

n_time, n_channels = traces.shape
print(f"Traces数据形状: ({n_time}, {n_channels})")

# 设置参数
sampling_rate = 20000  # 假设采样率为30000 Hz，可根据实际情况调整
time_window_sec = 5  # 20秒窗口
window_samples = int(time_window_sec * sampling_rate)
total_time_sec = n_time / sampling_rate  # 根据数据长度计算总时间
n_windows = int(np.ceil(total_time_sec / time_window_sec))

print(f"采样率: {sampling_rate} Hz")
print(f"总时间: {total_time_sec:.2f} 秒")
print(f"Time window: {time_window_sec}s ({window_samples} samples)")
print(f"总窗口数: {n_windows}")
print(f"总channel数: {n_channels}")

# 为每个channel分配不同的颜色
colors = plt.cm.tab20(np.linspace(0, 1, min(n_channels, 20)))

pdf_path_20s = "/media/ubuntu/sda/mouse_test/sorted/raw_trace.pdf"

with PdfPages(pdf_path_20s) as pdf:
    for window_idx in range(20):
        start_sample = window_idx * window_samples
        end_sample = min(start_sample + window_samples, traces.shape[0])
        
        if start_sample >= traces.shape[0]:
            break
        
        # 创建图形：n_channels行，1列
        fig, axes = plt.subplots(n_channels, 1, figsize=(12, n_channels * 0.3))
        
        # 设置整个图形的背景为透明
        fig.patch.set_alpha(0)
        fig.patch.set_facecolor('none')
        
        if n_channels == 1:
            axes = [axes]
        
        for channel_idx in range(n_channels):
            ax = axes[channel_idx]
            
            # 设置每个subplot的背景为透明
            ax.patch.set_alpha(0)
            ax.patch.set_facecolor('none')
            
            # 获取该channel的数据
            window_data = traces[start_sample:end_sample, channel_idx]
            
            # 转换为时间轴（秒）
            time_axis = np.arange(len(window_data)) / sampling_rate
            
            # 绘制，使用不同的颜色
            color_idx = channel_idx % len(colors)
            ax.plot(time_axis, window_data, color=colors[color_idx], linewidth=0.5)
            
            # 设置y轴范围（可以根据数据调整）
            y_min, y_max = window_data.min(), window_data.max()
            y_range = y_max - y_min
            if y_range > 0:
                ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.1)
            else:
                ax.set_ylim(-100, 100)
            
            # 设置标题（可选）
            ax.set_ylabel(f'Ch{channel_idx}', rotation=0, ha='right', va='center', fontsize=8)
            
            # 删除所有spines和ticks
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['bottom'].set_visible(False)
            ax.spines['left'].set_visible(False)
            ax.set_xticks([])
            ax.set_yticks([])
        
        # 只在最后一个subplot显示x轴标签
        if window_idx == n_windows - 1:
            axes[-1].spines['bottom'].set_visible(True)
            axes[-1].set_xlabel('Time (s)', fontsize=10)
            axes[-1].set_xticks([0, time_window_sec])
        
        plt.suptitle(f'Window {window_idx + 1}/{n_windows} ({window_idx * time_window_sec:.1f}s - {min((window_idx + 1) * time_window_sec, total_time_sec):.1f}s)', 
                     fontsize=10, y=0.995)
        plt.tight_layout(pad=0.1)
        pdf.savefig(fig, bbox_inches='tight', pad_inches=0, transparent=True, facecolor='none')
        plt.close(fig)
        
        if (window_idx + 1) % 1 == 0:
            print(f"  已生成 {window_idx + 1}/{n_windows} 页")

print(f"✓ 保存: {pdf_path_20s}")


生成20s窗口PDF (100s数据, 每个窗口20s)
Traces数据形状: (5999000, 20)
采样率: 20000 Hz
总时间: 299.95 秒
Time window: 5s (100000 samples)
总窗口数: 60
总channel数: 20
  已生成 1/60 页
  已生成 2/60 页
  已生成 3/60 页
  已生成 4/60 页
  已生成 5/60 页
  已生成 6/60 页
  已生成 7/60 页
  已生成 8/60 页
  已生成 9/60 页
  已生成 10/60 页
  已生成 11/60 页
  已生成 12/60 页
  已生成 13/60 页
  已生成 14/60 页
  已生成 15/60 页
  已生成 16/60 页
  已生成 17/60 页
  已生成 18/60 页
  已生成 19/60 页
  已生成 20/60 页
✓ 保存: /media/ubuntu/sda/mouse_test/sorted/raw_trace.pdf


In [11]:
# 绘制600 ms窗口的raw trace（使用已存在的 traces）
assert 'traces' in locals() or 'traces' in globals(), "请先准备好 traces (形状: n_time x n_channels)"
assert traces.ndim == 2, f"traces 应为2维数组，当前形状: {traces.shape}"

n_time, n_channels = traces.shape
sampling_rate = 20000  # 如有需要可修改为实际采样率

# 参数：每页600 ms
time_window_sec = 0.1
window_samples = int(time_window_sec * sampling_rate)
total_time_sec = n_time / sampling_rate
n_windows = int(np.ceil(total_time_sec / time_window_sec))

print("\n" + "="*60)
print(f"绘制600 ms窗口的raw trace | 总时间: {total_time_sec:.2f}s | 通道数: {n_channels}")
print("="*60)

colors = plt.cm.tab20(np.linspace(0, 1, min(n_channels, 20)))
pdf_path_600ms = "/media/ubuntu/sda/mouse_test/sorted/raw_trace_600ms.pdf"

with PdfPages(pdf_path_600ms) as pdf:
    for window_idx in range(30):
        start_sample = window_idx * window_samples
        end_sample = min(start_sample + window_samples, n_time)
        if start_sample >= n_time:
            break

        fig, axes = plt.subplots(n_channels, 1, figsize=(10, n_channels * 0.28))
        if n_channels == 1:
            axes = [axes]

        for ch in range(n_channels):
            ax = axes[ch]
            window_data = traces[start_sample:end_sample, ch]
            time_axis = np.arange(len(window_data)) / sampling_rate
            color_idx = ch % len(colors)
            ax.plot(time_axis, window_data, color=colors[color_idx], linewidth=1.5)

            y_min, y_max = window_data.min(), window_data.max()
            y_range = y_max - y_min
            if y_range > 0:
                ax.set_ylim(y_min - 0.1 * y_range, y_max + 0.1 * y_range)
            else:
                ax.set_ylim(-100, 100)

            ax.set_ylabel(f'Ch{ch}', rotation=0, ha='right', va='center', fontsize=7)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['bottom'].set_visible(False)
            ax.spines['left'].set_visible(False)
            ax.set_xticks([])
            ax.set_yticks([])

        axes[-1].spines['bottom'].set_visible(True)
        axes[-1].set_xlabel('Time (s)', fontsize=9)
        axes[-1].set_xticks([0, time_window_sec])

        plt.suptitle(f'Window {window_idx + 1}/{n_windows} ({start_sample/sampling_rate:.3f}s - {end_sample/sampling_rate:.3f}s)', fontsize=10, y=0.995)
        plt.tight_layout(pad=0.1)
        pdf.savefig(fig, bbox_inches='tight', pad_inches=0, transparent=True, facecolor='none')
        plt.close(fig)

        if (window_idx + 1) % 10 == 0 or window_idx == n_windows - 1:
            print(f"  已生成 {window_idx + 1}/{n_windows} 页")

print(f"✓ 600 ms trace PDF 已保存: {pdf_path_600ms}")



绘制600 ms窗口的raw trace | 总时间: 299.95s | 通道数: 20
  已生成 10/3000 页
  已生成 20/3000 页
  已生成 30/3000 页
✓ 600 ms trace PDF 已保存: /media/ubuntu/sda/mouse_test/sorted/raw_trace_600ms.pdf


In [27]:
import pickle 
with open("/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351/neuron_inf.pkl", 'rb') as f:
    neuron_inf = pickle.load(f)

In [40]:
waveform[i, :50].shape

(50,)

In [46]:
pdf_path_600ms = "/media/ubuntu/sda/mouse_test/sorted/waveform.pdf"
waveform = np.stack(neuron_inf['position_waveform'].values)
max_amplitude = waveform.max()
min_amplitude = waveform.min()

colors = plt.cm.tab20(np.linspace(0, 1, min(len(neuron_inf), 20)))
with PdfPages(pdf_path_600ms) as pdf:
    for i in range(len(neuron_inf)):
        plt.figure(figsize=(4, 2))
        plt.plot(np.arange(50), waveform[i, :50], color=colors[ i % len(colors)], linewidth=4)
        plt.ylim(min_amplitude, max_amplitude)
        pdf.savefig()
        plt.close()

In [35]:
colors[i // 20]

array([1.        , 0.49803922, 0.05490196, 1.        ])